# 🐍 Python for Everybody — Course 4: Using Databases with Python

> **Platform:** Coursera | **Instructor:** Dr. Charles Severance (University of Michigan)  
> **Level:** Intermediate | **Language:** Python 3

---

## 📋 About This Course

This course introduces relational databases and SQL using SQLite and Python. It covers creating tables, inserting and querying data, multi-table design with foreign keys, and processing real-world data from files (text, XML, JSON) directly into a database.

---

## 📚 Table of Contents

| Week | Topic | Concepts |
|------|-------|----------|
| Week 2 | Email Database | SQLite, CREATE/INSERT/UPDATE, SELECT ORDER BY |
| Week 3 | Music Library DB | Multi-table design, XML parsing, JOIN queries |
| Week 4 | Course Roster DB | JSON parsing, 3-table relational model, many-to-many |
| Bonus | Tracks (Simplified) | XML to SQLite pipeline, Artist/Album/Track schema |

---

## 📌 Week 2 — Email Organization Database

### Assignment Description
Read `mbox.txt`, extract the domain from each `From:` line, and store the email count per domain in a SQLite database. Then query and print results sorted by count descending.

### Concepts Covered
- **`sqlite3.connect()`** — create/connect to a SQLite database file
- **`CREATE TABLE`** — define a table schema
- **`INSERT / UPDATE`** — add or modify rows
- **`SELECT ... ORDER BY ... DESC`** — query and sort results
- **`cur.fetchone()`** — retrieve a single row from a query

### Key Takeaway
> Databases persist data between runs and allow powerful queries that would be complex with plain Python dicts. The INSERT-or-UPDATE pattern here is the foundation of any data ingestion pipeline.

> 📁 **Required file:** `mbox.txt` (included in this course folder)

In [ ]:
# Week 2 — Count emails per domain and store in SQLite
import sqlite3

conn = sqlite3.connect('emaildb.sqlite')
cur = conn.cursor()

cur.execute('DROP TABLE IF EXISTS Counts')
cur.execute('CREATE TABLE Counts (org TEXT, count INTEGER)')

fname = input('Enter file name: ')  # use: mbox.txt
if len(fname) < 1: fname = 'mbox.txt'
fh = open(fname)

for line in fh:
    if not line.startswith('From: '): continue
    pieces = line.split()
    email = pieces[1]
    x = email.find('@')
    dom = email[x+1:]
    cur.execute('SELECT count FROM Counts WHERE org = ?', (dom,))
    row = cur.fetchone()
    if row is None:
        cur.execute('INSERT INTO Counts (org, count) VALUES (?, 1)', (dom,))
    else:
        cur.execute('UPDATE Counts SET count = count + 1 WHERE org = ?', (dom,))

conn.commit()

sqlstr = 'SELECT org, count FROM Counts ORDER BY count DESC'
for row in cur.execute(sqlstr):
    print(str(row[0]), row[1])

cur.close()

iupui.edu 536
umich.edu 491
indiana.edu 178
caret.cam.ac.uk 157
uct.ac.za 92
media.berkeley.edu 56
(...)


## 📌 Week 3 — Music Library Multi-Table Database

### Assignment Description
Parse an iTunes XML file (`Library.xml`), extract track metadata, and populate a relational database with 4 linked tables: Artist, Album, Track, and Genre.

### Database Schema
```
Artist ──< Album ──< Track >── Genre
```

### Concepts Covered
- **Multi-table schema** — Artist, Album, Track, Genre with foreign keys
- **`INSERT OR IGNORE`** — avoid duplicate entries
- **`INSERT OR REPLACE`** — upsert pattern
- **XML traversal** — custom `lookup()` helper for iTunes dict format
- **JOIN query** — combining 4 tables in a single SQL query

### SQL Query to verify results
```sql
SELECT Track.title, Artist.name, Album.title, Genre.name
    FROM Track JOIN Genre JOIN Album JOIN Artist
    ON Track.genre_id = Genre.id AND Track.album_id = Album.id
        AND Album.artist_id = Artist.id
    ORDER BY Artist.name, Track.title LIMIT 3
```

> 📁 **Required files:** `Library.xml`, `week3.sqlite` (included in this course folder)

In [ ]:
# Week 3 — Parse iTunes XML into a 4-table relational database
import xml.etree.ElementTree as ET
import sqlite3

conn = sqlite3.connect('week3.sqlite')
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS Artist;
DROP TABLE IF EXISTS Album;
DROP TABLE IF EXISTS Track;
DROP TABLE IF EXISTS Genre;

CREATE TABLE Artist (
    id   INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name TEXT UNIQUE
);
CREATE TABLE Album (
    id        INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    artist_id INTEGER,
    title     TEXT UNIQUE
);
CREATE TABLE Track (
    id       INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    title    TEXT UNIQUE,
    album_id INTEGER,
    genre_id INTEGER,
    len INTEGER, rating INTEGER, count INTEGER
);
CREATE TABLE Genre (
    id   INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name TEXT UNIQUE
);
''')

fname = input('Enter file name: ')  # use: Library.xml
if len(fname) < 1: fname = 'Library.xml'

def lookup(d, key):
    found = False
    for child in d:
        if found: return child.text
        if child.tag == 'key' and child.text == key:
            found = True
    return None

stuff = ET.parse(fname)
all_entries = stuff.findall('dict/dict/dict')
print('Dict count:', len(all_entries))

for entry in all_entries:
    if lookup(entry, 'Track ID') is None: continue
    name   = lookup(entry, 'Name')
    artist = lookup(entry, 'Artist')
    album  = lookup(entry, 'Album')
    count  = lookup(entry, 'Play Count')
    rating = lookup(entry, 'Rating')
    length = lookup(entry, 'Total Time')
    genre  = lookup(entry, 'Genre')
    if name is None or artist is None or album is None or genre is None: continue

    print(name, artist, album, count, rating, length, genre)

    cur.execute('INSERT OR IGNORE INTO Artist (name) VALUES (?)', (artist,))
    cur.execute('SELECT id FROM Artist WHERE name = ?', (artist,))
    artist_id = cur.fetchone()[0]

    cur.execute('INSERT OR IGNORE INTO Genre (name) VALUES (?)', (genre,))
    cur.execute('SELECT id FROM Genre WHERE name = ?', (genre,))
    genre_id = cur.fetchone()[0]

    cur.execute('INSERT OR IGNORE INTO Album (title, artist_id) VALUES (?, ?)', (album, artist_id))
    cur.execute('SELECT id FROM Album WHERE title = ?', (album,))
    album_id = cur.fetchone()[0]

    cur.execute('''INSERT OR REPLACE INTO Track
        (title, album_id, len, rating, count, genre_id)
        VALUES (?, ?, ?, ?, ?, ?)''',
        (name, album_id, length, rating, count, genre_id))

    conn.commit()

Dict count: 404
Another One Bites The Dust Queen Greatest Hits 55 100 217103 Rock
Asche Zu Asche Rammstein Herzeleid None None 231810 Rock
Beauty School Dropout Various Grease None None 153000 Soundtrack
(...)


## 📌 Week 4 — Course Roster Many-to-Many Database

### Assignment Description
Read a JSON roster file containing `[name, course, role]` entries. Populate a 3-table relational database (User, Course, Member) that models a many-to-many relationship between students and courses.

### Database Schema
```
User ──< Member >── Course
         (role)
```

### Concepts Covered
- **Many-to-many relationships** — junction table `Member` with composite primary key
- **JSON data ingestion** — `json.loads()` into a database
- **`INSERT OR IGNORE`** — safely handle duplicate users/courses
- **`INSERT OR REPLACE`** — upsert memberships
- **Role field** — 1 = instructor, 0 = student

> 📁 **Required file:** `roster_data_sample.json` (included in this course folder)

In [ ]:
# Week 4 — Course Roster: JSON to 3-table relational database
import json
import sqlite3

conn = sqlite3.connect('rosterdb.sqlite')
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS User;
DROP TABLE IF EXISTS Member;
DROP TABLE IF EXISTS Course;

CREATE TABLE User (
    id   INTEGER NOT NULL PRIMARY KEY UNIQUE,
    name TEXT UNIQUE
);
CREATE TABLE Course (
    id    INTEGER NOT NULL PRIMARY KEY UNIQUE,
    title TEXT UNIQUE
);
CREATE TABLE Member (
    user_id   INTEGER,
    course_id INTEGER,
    role      INTEGER,
    PRIMARY KEY (user_id, course_id)
);
''')

fname = input('Enter file name: ')  # use: roster_data_sample.json
if len(fname) < 1: fname = 'roster_data_sample.json'

json_data = json.loads(open(fname).read())

for entry in json_data:
    name, title, role = entry[0], entry[1], entry[2]
    print((name, title, role))

    cur.execute('INSERT OR IGNORE INTO User (name) VALUES (?)', (name,))
    cur.execute('SELECT id FROM User WHERE name = ?', (name,))
    user_id = cur.fetchone()[0]

    cur.execute('INSERT OR IGNORE INTO Course (title) VALUES (?)', (title,))
    cur.execute('SELECT id FROM Course WHERE title = ?', (title,))
    course_id = cur.fetchone()[0]

    cur.execute('INSERT OR REPLACE INTO Member (user_id, course_id, role) VALUES (?, ?, ?)',
                (user_id, course_id, role))

conn.commit()

('Charley', 'si110', 1)
('Mea', 'si110', 0)
('Hattie', 'si110', 0)
('Lynh', 'si110', 0)
('Keziah', 'si110', 0)
(...)


## 📌 Bonus — Tracks: Simplified XML to SQLite Pipeline

### Description
A simplified version of the Week 3 music database — uses only 3 tables (Artist, Album, Track) without Genre. Demonstrates the core XML-to-SQLite pipeline in a cleaner form.

### Concepts Covered
- **3-table schema** — Artist, Album, Track with foreign keys
- **`lookup()` helper** — navigating iTunes XML dict format
- **`INSERT OR REPLACE`** — full upsert on Track entries

> 📁 **Required files:** `Library.xml`, `trackdb.sqlite` (included in this course folder)

In [ ]:
# Bonus — Simplified XML to SQLite: Artist / Album / Track
import xml.etree.ElementTree as ET
import sqlite3

conn = sqlite3.connect('trackdb.sqlite')
cur = conn.cursor()

cur.executescript('''
DROP TABLE IF EXISTS Artist;
DROP TABLE IF EXISTS Album;
DROP TABLE IF EXISTS Track;

CREATE TABLE Artist (
    id   INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    name TEXT UNIQUE
);
CREATE TABLE Album (
    id        INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    artist_id INTEGER,
    title     TEXT UNIQUE
);
CREATE TABLE Track (
    id       INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT UNIQUE,
    title    TEXT UNIQUE,
    album_id INTEGER,
    len INTEGER, rating INTEGER, count INTEGER
);
''')

fname = input('Enter file name: ')  # use: Library.xml
if len(fname) < 1: fname = 'Library.xml'

def lookup(d, key):
    found = False
    for child in d:
        if found: return child.text
        if child.tag == 'key' and child.text == key:
            found = True
    return None

stuff = ET.parse(fname)
all_entries = stuff.findall('dict/dict/dict')
print('Dict count:', len(all_entries))

for entry in all_entries:
    if lookup(entry, 'Track ID') is None: continue
    name   = lookup(entry, 'Name')
    artist = lookup(entry, 'Artist')
    album  = lookup(entry, 'Album')
    count  = lookup(entry, 'Play Count')
    rating = lookup(entry, 'Rating')
    length = lookup(entry, 'Total Time')
    if name is None or artist is None or album is None: continue

    print(name, artist, album, count, rating, length)

    cur.execute('INSERT OR IGNORE INTO Artist (name) VALUES (?)', (artist,))
    cur.execute('SELECT id FROM Artist WHERE name = ?', (artist,))
    artist_id = cur.fetchone()[0]

    cur.execute('INSERT OR IGNORE INTO Album (title, artist_id) VALUES (?, ?)', (album, artist_id))
    cur.execute('SELECT id FROM Album WHERE title = ?', (album,))
    album_id = cur.fetchone()[0]

    cur.execute('''INSERT OR REPLACE INTO Track
        (title, album_id, len, rating, count) VALUES (?, ?, ?, ?, ?)''',
        (name, album_id, length, rating, count))

    conn.commit()

Dict count: 404
Another One Bites The Dust Queen Greatest Hits 55 100 217103
Asche Zu Asche Rammstein Herzeleid None None 231810
Beauty School Dropout Various Grease None None 153000
(...)


---

## ✅ Course Summary

| Concept | Used In |
|---|---|
| SQLite connection & cursor | Week 2, 3, 4, Bonus |
| CREATE / INSERT / UPDATE / SELECT | Week 2 |
| Multi-table schema with foreign keys | Week 3, 4, Bonus |
| XML parsing into database | Week 3, Bonus |
| JSON parsing into database | Week 4 |
| Many-to-many relationships | Week 4 |
| INSERT OR IGNORE / REPLACE (upsert) | Week 3, 4, Bonus |

> 🎓 **Certificate:** [Python for Everybody Specialization — Coursera](https://www.coursera.org/specializations/python)